In [1]:
!pip install pyspark

In [38]:
import os
import glob
import platform
# --- Ensure a JVM is available for PySpark (Spark 4 needs Java 17+) ---
# Auto-detect a local JDK on macOS, Linux OR Windows if JAVA_HOME isn't already set.
def _find_java_home():
    jh = os.environ.get("JAVA_HOME")
    if jh and os.path.isdir(jh):
        return jh
    system = platform.system()
    candidates = []
    if system == "Darwin":            # macOS (Homebrew)
        candidates += ["/opt/homebrew/opt/openjdk@17", "/opt/homebrew/opt/openjdk@21",
                       "/opt/homebrew/opt/openjdk", "/usr/local/opt/openjdk@17"]
    elif system == "Windows":         # common Windows JDK install locations
        for base in (r"C:\Program Files\Eclipse Adoptium",
                     r"C:\Program Files\Java",
                     r"C:\Program Files\Microsoft\jdk-17",
                     r"C:\Program Files\Amazon Corretto",
                     r"C:\Program Files\Zulu"):
            candidates += sorted(glob.glob(base + r"*17*"))
            candidates += sorted(glob.glob(base + r"*21*"))
            candidates += sorted(glob.glob(base + r"*"))
    else:                             # Linux
        for pat in ("*17*", "*21*", "*"):
            candidates += sorted(glob.glob(f"/usr/lib/jvm/{pat}"))
    exe = "java.exe" if system == "Windows" else "java"
    for c in candidates:
        if os.path.isfile(os.path.join(c, "bin", exe)):
            return c
    return None

_jh = _find_java_home()
if _jh:
    os.environ["JAVA_HOME"] = _jh
    os.environ["PATH"] = os.path.join(_jh, "bin") + os.pathsep + os.environ["PATH"]
    print("JAVA_HOME =", _jh)
else:
    print("No JDK found automatically — install Java 17+ and set JAVA_HOME manually.")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("read-netflix-sources")
    # Force Spark onto loopback so it doesn't bind to a LAN/VPN address
    # (avoids "Connection reset" when initialising the local driver on macOS).
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    # MySQL JDBC driver on the classpath (downloaded from Maven on first run).
    # This must be set BEFORE the session starts — restart the kernel if you change it.
    .config("spark.jars.packages", "com.mysql:mysql-connector-j:9.1.0")
    .getOrCreate()
)
spark

JAVA_HOME = /usr/lib/jvm/java-1.17.0-openjdk-amd64


## Extracting The data from CSV file

In [4]:
CSV_PATH = "/content/netflix_titles.csv"

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print("Dataset loaded successfully")
print("Total rows:", raw.count())
print("Total columns:", len(raw.columns))

Dataset loaded successfully
Total rows: 8809
Total columns: 12


In [6]:
raw.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [39]:
raw.show(5, truncate=False)

+-------+-------+---------------------+---------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+------------------+------------+------+---------+-------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|show_id|type   |title                |director       |cast                                                                                                                                                                                                                                                                                                           |cou

In [8]:
row_count = raw.count()
column_count = len(raw.columns)

print("Dataset Dimensions")
print("-------------------")
print("Total Rows:", row_count)
print("Total Columns:", column_count)

Dataset Dimensions
-------------------
Total Rows: 8809
Total Columns: 12


In [9]:
from pyspark.sql import functions as F

null_counts = raw.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in raw.columns
])

print("Null Values in Each Column:")
null_counts.show(truncate=False)

Null Values in Each Column:
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|0      |1   |2    |2636    |826 |832    |13        |2           |6     |5       |3        |3          |
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



## Tranformation Chain

In [11]:
clean = (
    raw

    # 1. Drop rows with no title and remove duplicate IDs
    .dropna(subset=["show_id", "title"])
    .dropDuplicates(["show_id"])

    # 2. Fill missing categorical values
    .na.fill({
        "country": "Unknown",
        "director": "Unknown",
        "cast": "Unknown",
        "rating": "Not Rated",
        "listed_in": "Unknown"
    })

    # 3. Fix data types
    .withColumn(
        "release_year",
        F.col("release_year").cast("int")
    )
    .withColumn(
        "date_added",
        F.to_date(
            F.trim(F.col("date_added")),
            "MMMM d, yyyy"
        )
    )

    # 4. Derive helpful columns
    .withColumn(
        "content_age",
        F.year(F.current_date()) - F.col("release_year")
    )
    .withColumn(
        "primary_country",
        F.trim(
            F.split(F.col("country"), ",").getItem(0)
        )
    )

    # 5. Explode multi-value genres into one row per genre
    .withColumn(
        "genre",
        F.explode(
            F.split(F.col("listed_in"), ",\\s*")
        )
    )
)

print("Transformation chain defined successfully.")

Transformation chain defined successfully.


In [15]:
print("Raw rows:", raw.count(), " Clean rows:", clean.count())

clean.select(
    "title",
    "type",
    "primary_country",
    "release_year",
    "content_age",
    "genre"
).show(10, truncate=False)

Raw rows: 8809  Clean rows: 19304
+--------------------+-------+---------------+------------+-----------+--------------------+
|title               |type   |primary_country|release_year|content_age|genre               |
+--------------------+-------+---------------+------------+-----------+--------------------+
|Dick Johnson Is Dead|Movie  |United States  |2020        |6          |Documentaries       |
|The Starling        |Movie  |United States  |2021        |5          |Comedies            |
|The Starling        |Movie  |United States  |2021        |5          |Dramas              |
|On the Verge        |TV Show|France         |2021        |5          |TV Comedies         |
|On the Verge        |TV Show|France         |2021        |5          |TV Dramas           |
|Stowaway            |Movie  |Germany        |2021        |5          |Dramas              |
|Stowaway            |Movie  |Germany        |2021        |5          |International Movies|
|Stowaway            |Movie  |German

##Loading into MYSQL Server and Validating it

In [16]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y mysql-server > /dev/null 2>&1

print("MySQL Server installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
MySQL Server installed


In [17]:
!service mysql start
!service mysql status

 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.
 * /usr/bin/mysqladmin  Ver 8.0.46-0ubuntu0.22.04.3 for Linux on x86_64 ((Ubuntu))
Copyright (c) 2000, 2026, Oracle and/or its affiliates.

Oracle is a registered trademark of Oracle Corporation and/or its
affiliates. Other names may be trademarks of their respective
owners.

Server version		8.0.46-0ubuntu0.22.04.3
Protocol version	10
Connection		Localhost via UNIX socket
UNIX socket		/var/run/mysqld/mysqld.sock
Uptime:			2 sec

Threads: 2  Questions: 8  Slow queries: 0  Opens: 119  Flush tables: 3  Open tables: 38  Queries per second avg: 4.000


In [18]:
!mysql -e "CREATE DATABASE IF NOT EXISTS netflix_db;"

print("Database created successfully")

Database created successfully


In [19]:
!pip install -q mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 32.2 MB/s eta 0:00:00


In [22]:
import subprocess

sql_commands = """
CREATE DATABASE IF NOT EXISTS netflix_db;

CREATE USER IF NOT EXISTS 'colab_user'@'localhost'
IDENTIFIED BY 'colab123';

GRANT ALL PRIVILEGES
ON netflix_db.*
TO 'colab_user'@'localhost';

FLUSH PRIVILEGES;
"""

result = subprocess.run(
    ["mysql", "-u", "root"],
    input=sql_commands,
    text=True,
    capture_output=True
)

if result.returncode == 0:
    print("Database and MySQL user created successfully")
else:
    print("Error:")
    print(result.stderr)

Database and MySQL user created successfully


In [23]:
import mysql.connector

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="colab_user",
    password="colab123",
    database="netflix_db"
)

cursor = conn.cursor()

print("Connected to local MySQL successfully")

Connected to local MySQL successfully


In [24]:
# Check that the cleaned PySpark DataFrame still exists

print("Clean DataFrame rows:", clean.count())
print("Clean DataFrame columns:", len(clean.columns))

clean.printSchema()
clean.show(5, truncate=False)

Clean DataFrame rows: 19304
Clean DataFrame columns: 15
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = false)
 |-- cast: string (nullable = false)
 |-- country: string (nullable = false)
 |-- date_added: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = false)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = false)
 |-- description: string (nullable = true)
 |-- content_age: integer (nullable = true)
 |-- primary_country: string (nullable = true)
 |-- genre: string (nullable = false)

+-------+-----+------------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+----------+------------+------+--------+---------------------------------------+-----------------

In [33]:
from pyspark.sql import functions as F

clean_safe = (
    raw
    .dropna(
        thresh=2,
        subset=["show_id", "title"]
    )

    .dropDuplicates(["show_id"])

    .fillna({
        "director": "Unknown",
        "cast": "Unknown",
        "country": "Unknown",
        "rating": "Not Rated",
        "listed_in": "Unknown"
    })

    .withColumn(
        "release_year",
        F.col("release_year").cast("int")
    )

    .withColumn(
        "date_added",
        F.to_date(
            F.expr(
                "try_to_timestamp("
                "trim(date_added), "
                "'MMMM d, yyyy'"
                ")"
            )
        )
    )

    .withColumn(
        "content_age",
        F.year(F.current_date())
        - F.col("release_year")
    )

    .withColumn(
        "primary_country",
        F.trim(
            F.split(
                F.col("country"),
                ","
            ).getItem(0)
        )
    )

    .withColumn(
        "genre",
        F.explode(
            F.split(
                F.col("listed_in"),
                r",\s*"
            )
        )
    )
)

print("Testing full DataFrame...")

print(
    "Rows:",
    clean_safe.count()
)

print(
    "Columns:",
    len(clean_safe.columns)
)

clean_safe.show(
    5,
    truncate=False
)

Testing full DataFrame...
Rows: 19304
Columns: 15
+-------+-----+------------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+----------+------------+------+--------+---------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+--------------------+
|show_id|type |title       |director      |cast                                                                                                                                                                  |country               |date_added|release_year|rating|duration|listed_in                              |description                                                                                                               

In [34]:
# Collect the safely cleaned PySpark data
rows = clean_safe.collect()

insert_query = """
INSERT INTO netflix_clean (
    show_id,
    type,
    title,
    director,
    cast,
    country,
    date_added,
    release_year,
    rating,
    duration,
    listed_in,
    description,
    content_age,
    primary_country,
    genre
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

data_to_insert = [
    tuple(row[col] for col in clean_safe.columns)
    for row in rows
]

cursor.executemany(
    insert_query,
    data_to_insert
)

conn.commit()

print("Clean data uploaded successfully to MySQL")
print("Rows inserted:", len(data_to_insert))

Clean data uploaded successfully to MySQL
Rows inserted: 19304


In [35]:
# Source count from cleaned PySpark DataFrame
spark_row_count = clean_safe.count()

# Target count from MySQL
cursor.execute(
    "SELECT COUNT(*) FROM netflix_clean"
)

mysql_row_count = cursor.fetchone()[0]

print("ROW COUNT VALIDATION")
print("-" * 40)

print(
    "PySpark clean rows :",
    spark_row_count
)

print(
    "MySQL table rows   :",
    mysql_row_count
)

if spark_row_count == mysql_row_count:
    print("Status             : PASS")
    print("Row counts match successfully")
else:
    print("Status             : FAIL")
    print("Row counts do not match")

ROW COUNT VALIDATION
----------------------------------------
PySpark clean rows : 19304
MySQL table rows   : 19304
Status             : PASS
Row counts match successfully


In [36]:
# Source column count from PySpark
spark_column_count = len(clean_safe.columns)

# Target column count from MySQL
cursor.execute("""
    SELECT COUNT(*)
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'netflix_db'
      AND TABLE_NAME = 'netflix_clean'
""")

mysql_column_count = cursor.fetchone()[0]

print("COLUMN COUNT VALIDATION")
print("-" * 40)

print(
    "PySpark columns :",
    spark_column_count
)

print(
    "MySQL columns   :",
    mysql_column_count
)

if spark_column_count == mysql_column_count:
    print("Status          : PASS")
    print("Column counts match successfully")
else:
    print("Status          : FAIL")
    print("Column counts do not match")

COLUMN COUNT VALIDATION
----------------------------------------
PySpark columns : 15
MySQL columns   : 15
Status          : PASS
Column counts match successfully


In [45]:
import pandas as pd

query = """
SELECT *
FROM netflix_clean
LIMIT 5
"""

result = pd.read_sql(
    query,
    conn
)

result

/tmp/ipykernel_5394/4152035468.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,content_age,primary_country,genre
0,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,2021-09-24,2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...,5,United States,Comedies
1,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,2021-09-24,2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...,5,United States,Dramas
2,s1000,Movie,Stowaway,Joe Penna,"Anna Kendrick, Toni Collette, Daniel Dae Kim, ...","Germany, United States",2021-04-22,2021,TV-MA,116 min,"Dramas, International Movies, Thrillers",A three-person crew on a mission to Mars faces...,5,Germany,Dramas
3,s1000,Movie,Stowaway,Joe Penna,"Anna Kendrick, Toni Collette, Daniel Dae Kim, ...","Germany, United States",2021-04-22,2021,TV-MA,116 min,"Dramas, International Movies, Thrillers",A three-person crew on a mission to Mars faces...,5,Germany,International Movies
4,s1000,Movie,Stowaway,Joe Penna,"Anna Kendrick, Toni Collette, Daniel Dae Kim, ...","Germany, United States",2021-04-22,2021,TV-MA,116 min,"Dramas, International Movies, Thrillers",A three-person crew on a mission to Mars faces...,5,Germany,Thrillers
